In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 78 packages in 441ms
Checked 78 packages in 4ms


In [2]:
from dotenv import load_dotenv
import os
import sys
from openai import AzureOpenAI

# Load variables safely from your .env file
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = "2024-10-21"  # Explicitly set to match APIM routing
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-ptu")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Error: Missing settings in your .env file.")

print(f"Successfully loaded configuration. Key starts with: {azure_api_key[:6]}...")

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=api_version,
)
import gradio as gr

Successfully loaded configuration. Key starts with: 4e0a25...


c:\GroupProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
persona = (
    "You are the Main Support and Payment Specialist for Parking Services. "
    "You welcome users, help diagnose credit card processing errors, duplicate charges, "
    "and parking meter transaction failures, and manage payment lookups or refund requests safely."
)

# You can load your project documentation or sample parking rules here if applicable,
# or define a core system context for parking operations.
notes = """
Parking Services Customer Support Rules & Guidelines:
- Customers can experience gate transaction errors, duplicate charges, or meter payment failures.
- Payment Agent Tools allow looking up transaction records and submitting pending refunds.
- Refunds require transaction verification and cannot exceed the original charge amount.
"""

print("Persona Loaded:", persona)

Persona Loaded: You are the Main Support and Payment Specialist for Parking Services. You welcome users, help diagnose credit card processing errors, duplicate charges, and parking meter transaction failures, and manage payment lookups or refund requests safely.


In [4]:
import json

# Mock database simulating parking gate transactions and payment logs
TRANSACTION_DB = {
    "TXN-1001": {
        "gate": "Garage A North Gate", 
        "amount": 15.00, 
        "status": "Charged", 
        "refunded": False,
        "description": "Standard daily parking fee"
    },
    "TXN-1002": {
        "gate": "Garage B South Gate", 
        "amount": 25.00, 
        "status": "Double Charged", 
        "refunded": False,
        "description": "Duplicate gate terminal swipe error"
    },
    "TXN-1003": {
        "gate": "Surface Lot C Meter", 
        "amount": 8.50, 
        "status": "Charged", 
        "refunded": True,
        "description": "Overtime parking extension"
    }
}

def lookup_payment(transaction_id: str) -> str:
    """
    Finds the gate or meter transaction details from the database.
    
    Args:
        transaction_id (str): The unique transaction identifier (e.g., 'TXN-1001').
    """
    txn_id = transaction_id.strip().upper()
    txn = TRANSACTION_DB.get(txn_id)
    
    if not txn:
        return f"Error: Transaction ID '{transaction_id}' was not found in payment records."
    
    return json.dumps(txn)

def request_refund(transaction_id: str, amount: float, reason: str) -> str:
    """
    Submits a pending refund request. Enforces system guardrails: checks transaction existence,
    verifies amount does not exceed the charge, and prevents duplicate refunds[cite: 2].
    
    Args:
        transaction_id (str): The transaction ID to refund.
        amount (float): The monetary amount requested for refund.
        reason (str): The explanation for the refund request.
    """
    txn_id = transaction_id.strip().upper()
    txn_data = TRANSACTION_DB.get(txn_id)
    
    # Guardrail 1: Validate transaction exists
    if not txn_data:
        return f"Refund Failed: Transaction ID '{transaction_id}' is invalid or does not exist."
    
    # Guardrail 2: Prevent duplicate refunds
    if txn_data["refunded"]:
        return f"Refund Failed: Transaction '{transaction_id}' has already been refunded previously."
        
    # Guardrail 3: Verify amount does not exceed original charge
    if amount > txn_data["amount"]:
        return f"Refund Failed: Requested amount (${amount:.2f}) exceeds the original charge amount (${txn_data['amount']:.2f})."
        
    # Guardrail 4: High-limit check / approval routing (limits > $50 require human approval)
    if amount > 50.00:
        return f"PENDING HUMAN REVIEW: The refund request of ${amount:.2f} for transaction {txn_id} exceeds the automated threshold ($50.00) and has been routed to a human staff member for approval."

    # Process safe pending refund
    txn_data["refunded"] = True
    return f"SUCCESS: Pending refund of ${amount:.2f} for transaction {txn_id} successfully submitted and logged. Reason: {reason}"

In [5]:
lookup_payment_json = {
    "name": "lookup_payment",
    "description": "Finds a parking gate or meter transaction by its ID to check charges, status, and history.",
    "parameters": {
        "type": "object",
        "properties": {
            "transaction_id": {
                "type": "string",
                "description": "The unique transaction identifier (e.g., 'TXN-1001', 'TXN-1002')."
            }
        },
        "required": ["transaction_id"],
        "additionalProperties": False,
    },
}

request_refund_json = {
    "name": "request_refund",
    "description": "Submits a refund request for a verified parking transaction, enforcing amount limits, validation checks, and duplicate protection.",
    "parameters": {
        "type": "object",
        "properties": {
            "transaction_id": {
                "type": "string",
                "description": "The transaction ID to apply the refund to."
            },
            "amount": {
                "type": "number",
                "description": "The exact monetary amount to refund."
            },
            "reason": {
                "type": "string",
                "description": "The detailed explanation for why the refund is being requested."
            }
        },
        "required": ["transaction_id", "amount", "reason"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": lookup_payment_json},
    {"type": "function", "function": request_refund_json},
]

In [6]:
import json

# Mock database simulating parking gate transactions and payment logs
TRANSACTION_DB = {
    "TXN-1001": {
        "gate": "Garage A North Gate", 
        "amount": 15.00, 
        "status": "Charged", 
        "refunded": False,
        "description": "Standard daily parking fee"
    },
    "TXN-1002": {
        "gate": "Garage B South Gate", 
        "amount": 25.00, 
        "status": "Double Charged", 
        "refunded": False,
        "description": "Duplicate gate terminal swipe error"
    },
    "TXN-1003": {
        "gate": "Surface Lot C Meter", 
        "amount": 8.50, 
        "status": "Charged", 
        "refunded": True,
        "description": "Overtime parking extension"
    }
}

def lookup_payment(transaction_id: str) -> str:
    """
    Finds the gate or meter transaction details from the database.
    
    Args:
        transaction_id (str): The unique transaction identifier (e.g., 'TXN-1001').
    """
    txn_id = transaction_id.strip().upper()
    txn = TRANSACTION_DB.get(txn_id)
    
    if not txn:
        return f"Error: Transaction ID '{transaction_id}' was not found in payment records."
    
    return json.dumps(txn)

def request_refund(transaction_id: str, amount: float, reason: str) -> str:
    """
    Submits a pending refund request. Enforces system guardrails: checks transaction existence,
    verifies amount does not exceed the charge, and prevents duplicate refunds[cite: 2].
    
    Args:
        transaction_id (str): The transaction ID to refund.
        amount (float): The monetary amount requested for refund.
        reason (str): The explanation for the refund request.
    """
    txn_id = transaction_id.strip().upper()
    txn_data = TRANSACTION_DB.get(txn_id)
    
    # Guardrail 1: Validate transaction exists
    if not txn_data:
        return f"Refund Failed: Transaction ID '{transaction_id}' is invalid or does not exist."
    
    # Guardrail 2: Prevent duplicate refunds
    if txn_data["refunded"]:
        return f"Refund Failed: Transaction '{transaction_id}' has already been refunded previously."
        
    # Guardrail 3: Verify amount does not exceed original charge
    if amount > txn_data["amount"]:
        return f"Refund Failed: Requested amount (${amount:.2f}) exceeds the original charge amount (${txn_data['amount']:.2f})."
        
    # Guardrail 4: High-limit check / approval routing (limits > $50 require human approval)
    if amount > 50.00:
        return f"PENDING HUMAN REVIEW: The refund request of ${amount:.2f} for transaction {txn_id} exceeds the automated threshold ($50.00) and has been routed to a human staff member for approval."

    # Process safe pending refund
    txn_data["refunded"] = True
    return f"SUCCESS: Pending refund of ${amount:.2f} for transaction {txn_id} successfully submitted and logged. Reason: {reason}"

In [7]:
from IPython.display import Markdown, display
system_prompt = f"""
# Your role
You are {persona}

# Context
{notes}

# Rules
- Welcome the customer warmly and help diagnose any parking payment, receipt, or transaction problems[cite: 2].
- Call `lookup_payment` whenever a customer mentions a receipt, charge, or provides a transaction ID to inspect transaction records.
- Call `request_refund` whenever a customer explicitly asks for money back due to an error, duplicate charge, or gate malfunction, making sure you have checked the transaction details first[cite: 2].
- Maintain a helpful, empathetic, and professional support tone at all times.
"""

display(Markdown(system_prompt))


# Your role
You are You are the Main Support and Payment Specialist for Parking Services. You welcome users, help diagnose credit card processing errors, duplicate charges, and parking meter transaction failures, and manage payment lookups or refund requests safely.

# Context

Parking Services Customer Support Rules & Guidelines:
- Customers can experience gate transaction errors, duplicate charges, or meter payment failures.
- Payment Agent Tools allow looking up transaction records and submitting pending refunds.
- Refunds require transaction verification and cannot exceed the original charge amount.


# Rules
- Welcome the customer warmly and help diagnose any parking payment, receipt, or transaction problems[cite: 2].
- Call `lookup_payment` whenever a customer mentions a receipt, charge, or provides a transaction ID to inspect transaction records.
- Call `request_refund` whenever a customer explicitly asks for money back due to an error, duplicate charge, or gate malfunction, making sure you have checked the transaction details first[cite: 2].
- Maintain a helpful, empathetic, and professional support tone at all times.


In [8]:
def call_tool(name, arguments):
    if name == "lookup_payment":
        return lookup_payment(**arguments)
    elif name == "request_refund":
        return request_refund(**arguments)
    else:
        return f"Unknown tool: {name}"

In [9]:
import json

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            result = call_tool(name, arguments)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
